# 01 抓取 Statcast 逐球数据

## 这一步做了什么
用 `pybaseball.statcast()` 从 Baseball Savant 下载指定日期全部比赛的逐球（pitch-level）数据，
筛选出分析所需的关键列，并按日期保存为 CSV，供后续 Notebook 复用。

## 数据从哪来
- 数据源：MLB Baseball Savant（https://baseballsavant.mlb.com），由 `pybaseball` 封装
- 粒度：一球一行（一次投球 = 一行），包含球速、击球初速、击球仰角、打席结果等
- 抓取接口：`pybaseball.statcast(start_dt, end_dt)`

## 结果是什么
- 输出文件：`data/raw/2025-06-14_statcast.csv`
- 单日约 4000 行、约 25 列
- 注意：`events`、`launch_speed`、`launch_angle` 只在特定行有值（见下文说明），缺失是正常的


In [2]:
# ===== 环境准备：导入依赖 + 开启本地缓存 =====
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pybaseball as pb
from pybaseball import cache

# 1) 开启 pybaseball 本地缓存，同一请求第二次直接从磁盘读，不再走网络，对每天反复调试非常关键
cache.enable()

print("pybaseball 版本:", pb.__version__)
print("pandas 版本:", pd.__version__)


pybaseball 版本: 2.2.7
pandas 版本: 3.1.1


In [3]:
def fetch_statcast_day(date_str: str, max_retry: int = 3, wait: int = 10) -> pd.DataFrame:
    """抓取某一天全部比赛的 Statcast 逐球数据。
    参数
    ----
    date_str : str
        日期字符串，格式 'YYYY-MM-DD'
    max_retry : int
        网络失败时的最大重试次数
    wait : int
        重试基础等待秒数，按次数递增（10s / 20s / 30s），避免被服务器限流

    返回
    ----
    pd.DataFrame
        原始逐球数据（未筛选列）
    """
    for attempt in range(1, max_retry + 1):
        try:
            print(f"正在抓取 {date_str}（第 {attempt} 次尝试）...")
            df = pb.statcast(start_dt=date_str, end_dt=date_str, verbose=True)
            print(f"抓取成功，原始形状: {df.shape}")
            return df
        except Exception as e:
            print(f"[失败] {type(e).__name__}: {e}")
            if attempt < max_retry:
                time.sleep(wait * attempt)
    raise RuntimeError(
        f"{date_str} 抓取失败。请检查：1) 网络/代理是否可用；"
        f"2) 该日期是否确实有比赛；3) 改用离线 CSV 方案（见备用单元格）"
    )


In [4]:
# ===== 执行抓取 =====
DATE = "2025-06-14"
raw = fetch_statcast_day(DATE)

# 先看一眼最核心的 7 列长什么样
raw[["game_date", "pitcher", "batter", "release_speed",
     "events", "launch_speed", "launch_angle"]].head()


正在抓取 2025-06-14（第 1 次尝试）...
This is a large query, it may take a moment to complete


100%|██████████| 1/1 [00:11<00:00, 11.79s/it]

抓取成功，原始形状: (4435, 119)


,game_date,pitcher,batter,release_speed,events,launch_speed,launch_angle
2426,2025-06-14,669276,687597,87.9,strikeout,<NA>,<NA>
2546,2025-06-14,669276,687597,87.9,NaN,<NA>,<NA>
2578,2025-06-14,669276,687597,86.3,NaN,<NA>,<NA>
2700,2025-06-14,669276,687597,87.0,NaN,<NA>,<NA>
2751,2025-06-14,669276,687597,87.8,NaN,<NA>,<NA>


## 字段说明（保留列的理由）

Day 1 要求保留的 7 列：

| 列名 | 含义 | 缺失情况 |
|---|---|---|
| `game_date` | 比赛日期 | 无 |
| `pitcher` | 投手 MLBAM ID | 无 |
| `batter` | 打者 MLBAM ID | 无 |
| `release_speed` | 该球出手球速（mph） | 无 |
| `events` | 该球所在打席的最终结果 | **仅每个打席最后一球有值**，其余为空 |
| `launch_speed` | 击球初速（mph，exit velocity） | **仅打到球的行有值**，三振/保送/界外为空 |
| `launch_angle` | 击球仰角（度） | 同上 |

额外保留的上下文列（为 Day 2/Day 3 提前准备，避免重复爬取）：

- `game_pk` 比赛编号：合并多天数据、去重时用
- `home_team` / `away_team`：主客队
- `inning` / `inning_topbot`：局数与上下半局
- `outs_when_up`：该球投出前的出局数（Day 3 算得分期望的核心状态量）
- `on_1b` / `on_2b` / `on_3b`：该球投出前各垒上的跑者 ID，空表示该垒无人（Day 3 的另一个状态量）
- `at_bat_number` / `pitch_number`：打席与球序，便于去重和逐球追踪
- `description` / `type`：单球结果描述与 B/S/X 分类
- `bb_type`：击球类型（ground_ball / line_drive / fly_ball / popup）
- `stand` / `p_throws`：打者左右打、投手左右投


In [5]:
# ===== 筛选列 + 保存 CSV =====
KEEP_COLS = [
    # Day 1 要求的 7 列
    "game_date", "pitcher", "batter", "release_speed", "events", "launch_speed", "launch_angle",
    # 比赛定位
    "game_pk", "game_type", "home_team", "away_team",
    # 打席上下文（Day 3 必需）
    "inning", "inning_topbot", "outs_when_up", "on_1b", "on_2b", "on_3b",
    "at_bat_number", "pitch_number", "description", "type", "bb_type",
    "stand", "p_throws",
]

# 只保留数据里真实存在的列，缺失的列打印出来而不是直接报错
cols = [c for c in KEEP_COLS if c in raw.columns]
missing = [c for c in KEEP_COLS if c not in raw.columns]
if missing:
    print("以下列在本日数据中不存在，已跳过：", missing)

df = raw[cols].copy()
df["game_date"] = pd.to_datetime(df["game_date"]).dt.strftime("%Y-%m-%d")

# 保存到 data/raw/，目录不存在则自动创建
out_dir = Path("data/raw")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{DATE}_statcast.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"已保存: {out_path}")
print(f"形状: {df.shape}  |  文件大小: {out_path.stat().st_size / 1024:.0f} KB")


已保存: data\raw\2025-06-14_statcast.csv
形状: (4435, 24)  |  文件大小: 401 KB


In [6]:
# ===== 数据体检：确认抓到的不是空表或残缺表 =====
print("总行数（逐球）:", len(df))
print("日期:", df["game_date"].unique())
print("比赛场数:", df["game_pk"].nunique())
print("投手数:", df["pitcher"].nunique(), "| 打者数:", df["batter"].nunique())

# 缺失率——缺失不等于错误，要能解释为什么缺
print("\n各列缺失率（%）:")
print((df.isna().mean().sort_values(ascending=False) * 100).round(1).to_string())

# events 分布：验证盗垒、三振等事件是否都在（Day 2 要用）
print("\nevents 分布（前 12）:")
print(df["events"].value_counts(dropna=False).head(12).to_string())


总行数（逐球）: 4435
日期: <ArrowStringArray>
['2025-06-14']
Length: 1, dtype: str
比赛场数: 15
投手数: 130 | 打者数: 299

各列缺失率（%）:
on_3b            90.1
bb_type          83.2
on_2b            80.2
events           74.3
on_1b            70.0
launch_speed     67.0
launch_angle     66.9
release_speed     0.7
game_pk           0.0
game_date         0.0
pitcher           0.0
batter            0.0
inning            0.0
away_team         0.0
game_type         0.0
home_team         0.0
inning_topbot     0.0
outs_when_up      0.0
pitch_number      0.0
at_bat_number     0.0
description       0.0
type              0.0
stand             0.0
p_throws          0.0

events 分布（前 12）:
events
NaN                          3295
field_out                     462
strikeout                     262
single                        148
walk                          100
double                         45
home_run                       32
grounded_into_double_play      20
hit_by_pitch                   18
force_out                  

In [7]:
# ===== 把 MLBAM ID 转成球员姓名 =====
try:
    from pybaseball import playerid_reverse_lookup

    ids = pd.concat([df["pitcher"], df["batter"]]).dropna().astype(int).unique()
    lookup = playerid_reverse_lookup(ids, key_type="mlbam")
    lookup["name"] = (lookup["name_first"].str.title() + " " + lookup["name_last"].str.title())
    name_map = dict(zip(lookup["key_mlbam"], lookup["name"]))

    df["pitcher_name"] = df["pitcher"].map(name_map)
    df["batter_name"] = df["batter"].map(name_map)
    df.to_csv(out_path, index=False, encoding="utf-8-sig")   # 覆盖保存，带上姓名
    print(df[["pitcher_name", "batter_name", "release_speed"]].head().to_string())
except Exception as e:
    print("姓名映射跳过（不影响后续分析）：", type(e).__name__, e)


Gathering player lookup table. This may take a moment.
     pitcher_name  batter_name  release_speed
2426    Dylan Lee  Jordan Beck           87.9
2546    Dylan Lee  Jordan Beck           87.9
2578    Dylan Lee  Jordan Beck           86.3
2700    Dylan Lee  Jordan Beck           87.0
2751    Dylan Lee  Jordan Beck           87.8


In [8]:
# ===== 备用方案：手动下载的 Savant CSV 走同一条流水线 =====
def load_manual_savant_csv(path: str) -> pd.DataFrame:
    """读取从 Baseball Savant 网页手动导出的 CSV。

    Savant 导出的列名与 pybaseball 返回的完全一致，
    所以后续的 KEEP_COLS 筛选逻辑可以直接复用。
    """
    d = pd.read_csv(path, low_memory=False)
    d["game_date"] = pd.to_datetime(d["game_date"]).dt.strftime("%Y-%m-%d")
    return d

# 用法示例（把文件放到项目根目录后执行，然后跑单元格 6、7 即可）：
# raw = load_manual_savant_csv("statcast_2025-06-14.csv")
